In [3]:
from together import Together
from rdflib import Graph, RDF, RDFS, OWL, BNode
import csv
import time
import re
from pathlib import Path
import os
import sys

# ============================================================
# CONFIG
# ============================================================
CCO_TTL_PATH           = "CCO/CCO (V1).ttl"
MODEL_EVAL             = "deepseek-ai/DeepSeek-V3.1"
EVAL_LIMIT_PER_DOMAIN  = None
SAVE_EVERY             = 10
MAX_RETRIES            = 3
BASE_SLEEP_S           = 5
REQUEST_DELAY_S        = 0.3

DOMAINS = [
    ("Finance",         "Finance/cqs_finance_fixed.csv"),
    ("Healthcare",      "Healthcare/cqs_healthcare_fixed.csv"),
    ("Education",       "Education/cqs_education_fixed.csv"),
    ("Data Protection", "GDPR/cqs_gdpr_fixed.csv"),
]

EVAL_ONLY_COLS = [
    "cq_id", "domain", "clause", "excerpt",
    "question", "cco_elements",
    "llm_model", "llm_assessment", "llm_reason",
]

SYSTEM_MSG = (
    "You are a compliance ontology expert. "
    "You must respond only in English."
)

# ============================================================
# UTILITY FUNCTIONS
# ============================================================

def load_dotenv(dotenv_path=".env"):
    """Load environment variables from .env file if present."""
    p = Path(dotenv_path)
    if not p.exists():
        return
    for line in p.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k and v and k not in os.environ:
            os.environ[k] = v


def read_csv(path: Path) -> list:
    """Read CSV file and return list of row dicts."""
    with path.open("r", encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))


def write_csv(path: Path, rows: list, fieldnames: list) -> None:
    """Write list of row dicts to CSV file."""
    with path.open("w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        w.writeheader()
        for r in rows:
            w.writerow(r)


def sort_key_cqid(row: dict) -> int:
    """Return integer sort key extracted from cq_id field."""
    m = re.search(r"(\d+)", str(row.get("cq_id", "")))
    return int(m.group(1)) if m else 10 ** 12


def parse_eval(text: str):
    """Extract Assessment and Reason from LLM response."""
    text = text or ""
    a = re.search(r"Assessment:\s*(Yes|No|IDK)\b", text, re.IGNORECASE)
    r = re.search(r"Reason:\s*(.+?)\s*$", text, re.DOTALL)
    assessment = a.group(1).strip().capitalize() if a else ""
    reason     = r.group(1).strip() if r else ""
    return assessment, reason


# ============================================================
# AUTH
# ============================================================

load_dotenv(".env")

api_key = (os.environ.get("TOGETHER_API_KEY") or "").strip()
TOGETHER_API_KEY_HARDCODE = ""

if not api_key and TOGETHER_API_KEY_HARDCODE.strip():
    api_key = TOGETHER_API_KEY_HARDCODE.strip()

if not api_key:
    print("ERROR: Missing Together API key.")
    print("  1. Set env var: export TOGETHER_API_KEY='your_key'")
    print("  2. Add .env file: TOGETHER_API_KEY=your_key")
    print("  3. Set TOGETHER_API_KEY_HARDCODE in this script.")
    sys.exit(1)

client = Together(api_key=api_key)

# ============================================================
# SMOKE TEST
# ============================================================

def smoke_test() -> None:
    """Verify API connectivity before full run."""
    print("Running smoke test...")
    for attempt in range(1, 3):
        try:
            resp = client.chat.completions.create(
                model=MODEL_EVAL,
                messages=[
                    {"role": "system", "content": SYSTEM_MSG},
                    {"role": "user", "content": (
                        "Reply with exactly this and nothing else:\n"
                        "Assessment: Yes\n"
                        "Reason: smoke test passed"
                    )},
                ],
                max_tokens=40,
                temperature=0.0,
            )
            txt = resp.choices[0].message.content or ""
            a, _ = parse_eval(txt)
            if a == "Yes":
                print("Smoke test passed.")
            else:
                print(
                    f"WARNING: Unexpected smoke test response "
                    f"(continuing): {repr(txt[:100])}"
                )
            return
        except Exception as e:
            print(f"ERROR: Smoke test failed (attempt {attempt}): {e}")
            if attempt == 1:
                time.sleep(3)
            else:
                sys.exit(1)


smoke_test()

# ============================================================
# STEP 1 — Load CCO Schema
# ============================================================

def _local_name(uri: str) -> str:
    if "#" in uri:
        return uri.split("#")[-1]
    return uri.rstrip("/").split("/")[-1]


def get_restriction_label(g: Graph, node) -> str:
    on_property = g.value(node, OWL.onProperty)
    prop_label  = g.value(on_property, RDFS.label) if on_property else None
    prop_name   = (
        str(prop_label) if prop_label
        else (_local_name(str(on_property)) if on_property else "?")
    )
    some_values = g.value(node, OWL.someValuesFrom)
    on_class    = g.value(node, OWL.onClass)
    on_range    = g.value(node, OWL.onDataRange)
    min_card    = g.value(node, OWL.minQualifiedCardinality)
    max_card    = g.value(node, OWL.maxQualifiedCardinality)
    exact_card  = g.value(node, OWL.qualifiedCardinality)

    target = None
    if on_class:
        if isinstance(on_class, BNode):
            target = "[complex class expression]"
        else:
            tl = g.value(on_class, RDFS.label)
            target = str(tl) if tl else _local_name(str(on_class))
    elif on_range:
        target = _local_name(str(on_range))

    if some_values:
        if isinstance(some_values, BNode):
            ix = g.value(some_values, OWL.intersectionOf)
            return (
                f"{prop_name} some [complex intersection]" if ix
                else f"{prop_name} some [complex expression]"
            )
        sv = str(
            g.value(some_values, RDFS.label) or _local_name(str(some_values))
        )
        return f"{prop_name} some {sv}"

    if exact_card:
        return f"{prop_name} exactly {exact_card} {target or ''}".strip()
    if min_card:
        return f"{prop_name} min {min_card} {target or ''}".strip()
    if max_card:
        return f"{prop_name} max {max_card} {target or ''}".strip()
    return f"restriction on {prop_name}"


def load_cco_schema(ttl_path: str) -> str:
    """Load CCO ontology and return deterministic schema string."""
    g = Graph()
    g.parse(ttl_path, format="turtle")

    classes     = []
    constraints = {}
    obj_props   = []
    dat_props   = []

    for cls in g.subjects(RDF.type, OWL.Class):
        if not str(cls).startswith("http"):
            continue
        label    = g.value(cls, RDFS.label)
        cls_name = str(label) if label else _local_name(str(cls))
        cls_constraints = []
        super_names     = []

        for sc in g.objects(cls, RDFS.subClassOf):
            if isinstance(sc, BNode):
                ix = g.value(sc, OWL.intersectionOf)
                if ix:
                    for item in list(g.items(ix)):
                        if isinstance(item, BNode):
                            cls_constraints.append(
                                f"AND {get_restriction_label(g, item)}"
                            )
                else:
                    cls_constraints.append(get_restriction_label(g, sc))
            else:
                sl = g.value(sc, RDFS.label)
                super_names.append(
                    str(sl) if sl else _local_name(str(sc))
                )

        classes.append(cls_name)
        constraints[cls_name] = {
            "subclasses": super_names,
            "constraints": cls_constraints,
        }

    classes.sort()

    for prop in g.subjects(RDF.type, OWL.ObjectProperty):
        label  = g.value(prop, RDFS.label)
        domain = g.value(prop, RDFS.domain)
        range_ = g.value(prop, RDFS.range)
        obj_props.append({
            "name": str(label) if label else _local_name(str(prop)),
            "domain": (
                str(g.value(domain, RDFS.label) or _local_name(str(domain)))
                if domain else "None"
            ),
            "range": (
                str(g.value(range_, RDFS.label) or _local_name(str(range_)))
                if range_ else "None"
            ),
        })

    for prop in g.subjects(RDF.type, OWL.DatatypeProperty):
        label  = g.value(prop, RDFS.label)
        domain = g.value(prop, RDFS.domain)
        range_ = g.value(prop, RDFS.range)
        dat_props.append({
            "name": str(label) if label else _local_name(str(prop)),
            "domain": (
                str(g.value(domain, RDFS.label) or _local_name(str(domain)))
                if domain else "None"
            ),
            "range": _local_name(str(range_)) if range_ else "None",
        })

    schema_str = "CCO CLASSES (with constraints):\n"
    for cls_name in classes:
        info = constraints.get(cls_name, {})
        schema_str += f"- {cls_name}"
        if info.get("subclasses"):
            schema_str += f" (subClassOf: {', '.join(info['subclasses'])})"
        schema_str += "\n"
        for c in info.get("constraints", []):
            schema_str += f"    Constraint: {c}\n"

    schema_str += "\nOBJECT PROPERTIES (Domain -> Range):\n"
    for p in obj_props:
        schema_str += f"- {p['name']}: {p['domain']} -> {p['range']}\n"

    schema_str += "\nDATATYPE PROPERTIES (Domain -> Range):\n"
    for p in dat_props:
        schema_str += f"- {p['name']}: {p['domain']} -> {p['range']}\n"

    print(
        f"CCO schema loaded: {len(classes)} classes, "
        f"{len(obj_props)} object properties, "
        f"{len(dat_props)} datatype properties."
    )
    return schema_str


schema_str = load_cco_schema(CCO_TTL_PATH)

# ============================================================
# STEP 2 — Prompt Builder
# ============================================================

def build_eval_prompt(cq_row: dict, schema_str: str) -> str:
    """Build evaluation prompt including full CQ data."""
    return f"""You are a compliance ontology expert. Respond only in English.

Below is the schema of the Core Compliance Ontology (CCO). Use ONLY the listed classes and properties.

{schema_str}

Important interpretation notes (apply strictly):

1. STRUCTURAL REPRESENTABILITY only: Judge whether the normative structure
   (Regulation/Norm + deontic type + appliesToRole + hasAction/hasObject +
   appliesUnder/Condition + Exception patterns + temporal validity where
   explicit) can be captured using CCO.

2. Generic class coverage - do NOT mark No for domain terminology:
   - cco:Agent covers ALL actors: controllers, processors, data subjects,
     institutions, states, authorities, organisations, persons, bodies.
     Never mark No because a specific actor label is not a CCO class name.
   - cco:Role covers ANY named role. Never mark No because a specific role
     label is absent from CCO.
   - cco:Resource covers ANY asset, document, dataset, record, data, form,
     or object referenced in a norm.
   - cco:Action covers ANY required, permitted, or prohibited activity.
   - cco:Condition covers ANY trigger, qualifying circumstance, eligibility
     criterion, or deadline. Use cco:appliesUnder + cco:hasConditionExpression
     for the condition text.

3. External references are acceptable: References to external documents
   (e.g. "Article 74 of Directive X", "Articles 92 to 95", "Annex A") do
   NOT make a CQ invalid. CCO represents the normative structure, not the
   referenced document.

4. Mark No ONLY when:
   - The excerpt is purely definitional ("X means...") with NO deontic
     statement - no obligation, permission, or prohibition present.
   - The question asks for legal consequences ("what happens if...") that go
     beyond an explicit Exception/Condition structure in the excerpt.
   - The core normative structure fundamentally cannot be expressed in CCO
     even with generic class mapping.

5. Do NOT mark No because:
   - A domain-specific term is not literally a CCO class name.
   - An excerpt references another article or external directive.
   - The excerpt uses legal language not matching CCO property names exactly.

Competency question (CQ) grounded in a regulatory clause:

Question:        {cq_row.get('question', '')}
Clause:          {cq_row.get('clause', '')}
Regulatory Text: {cq_row.get('excerpt', '')}
CCO Elements:    {cq_row.get('cco_elements', '')}

Can this question be answered by representing the regulatory text using ONLY
the CCO classes and properties listed?

Respond strictly in this format:
Assessment: Yes/No/IDK
Reason: ... (one sentence)
"""

# ============================================================
# STEP 3 — LLM Call
# ============================================================

def call_llm(prompt: str, max_retries: int = MAX_RETRIES) -> str:
    """Call LLM with retry and rate limit handling."""
    for attempt in range(1, max_retries + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL_EVAL,
                messages=[
                    {"role": "system", "content": SYSTEM_MSG},
                    {"role": "user",   "content": prompt},
                ],
                max_tokens=180,
                temperature=0.0,
            )
            time.sleep(REQUEST_DELAY_S)
            return resp.choices[0].message.content or ""
        except Exception as e:
            err_str = str(e).lower()
            if "429" in err_str or "rate limit" in err_str:
                wait = 60 * attempt
                print(f"    Rate limit hit. Waiting {wait}s...")
                time.sleep(wait)
            else:
                print(f"    Attempt {attempt}/{max_retries} failed: {e}")
                if attempt < max_retries:
                    time.sleep(BASE_SLEEP_S * attempt)
    print("    All retries exhausted. Returning empty string.")
    return ""

# ============================================================
# STEP 4 — Main evaluation loop
# ============================================================

all_eval_only_rows = []

for domain_name, cq_file in DOMAINS:
    in_path = Path(cq_file)
    if not in_path.exists():
        print(f"SKIP: Input file not found: {cq_file}")
        continue

    # Temporary checkpoint file — deleted after domain completes
    checkpoint_path = in_path.with_name(
        in_path.stem + "_checkpoint_tmp.csv"
    )

    base_rows = read_csv(in_path)
    if not base_rows:
        print(f"SKIP: Empty input file: {cq_file}")
        continue

    # Resume from checkpoint if exists
    if checkpoint_path.exists():
        rows = read_csv(checkpoint_path)
        if len(rows) != len(base_rows):
            print(
                f"WARNING: {domain_name}: checkpoint row count mismatch. "
                f"Recreating from base."
            )
            rows = base_rows
        else:
            print(f"Resuming from checkpoint: {checkpoint_path}")
    else:
        rows = base_rows

    fieldnames = list(rows[0].keys())
    for col in ["llm_model", "llm_assessment", "llm_reason"]:
        if col not in fieldnames:
            fieldnames.append(col)

    pending = [
        r for r in rows
        if r.get("auto_keep") == "PASS"
        and not (r.get("llm_assessment") or "").strip()
    ]
    pending.sort(key=sort_key_cqid)

    to_eval = (
        pending[:EVAL_LIMIT_PER_DOMAIN]
        if EVAL_LIMIT_PER_DOMAIN is not None
        else pending
    )
    total = len(to_eval)

    print("\n" + "=" * 55)
    print(f"Domain:      {domain_name}")
    print(f"Pending:     {len(pending)} | Evaluating now: {total}")
    print(f"Input file:  {in_path}")
    print("=" * 55)

    for idx, row in enumerate(to_eval, start=1):
        prompt             = build_eval_prompt(row, schema_str)
        raw                = call_llm(prompt)
        assessment, reason = parse_eval(raw)

        row["llm_model"]      = MODEL_EVAL
        row["llm_assessment"] = assessment
        row["llm_reason"]     = reason

        pct          = (idx / total * 100) if total else 100.0
        short_reason = (reason[:75] + "...") if len(reason) > 75 else reason
        print(
            f"  {row.get('cq_id', '?')} "
            f"({idx}/{total}, {pct:.1f}%): "
            f"{assessment or 'EMPTY'} - {short_reason}"
        )

        # Save checkpoint periodically
        if idx % SAVE_EVERY == 0:
            write_csv(checkpoint_path, rows, fieldnames)
            print(f"  Checkpoint saved at {idx}")

    # Save final checkpoint
    write_csv(checkpoint_path, rows, fieldnames)

    done = sum(1 for r in rows if (r.get("llm_assessment") or "").strip())
    yes  = sum(1 for r in rows if r.get("llm_assessment") == "Yes")
    no   = sum(1 for r in rows if r.get("llm_assessment") == "No")
    idk  = sum(1 for r in rows if r.get("llm_assessment") == "IDK")

    print(f"\n{domain_name} complete.")
    print(f"  Total evaluated: {done} | Yes: {yes} | No: {no} | IDK: {idk}")

    # Collect slim rows for combined output
    for r in rows:
        if (r.get("llm_assessment") or "").strip():
            all_eval_only_rows.append(
                {c: (r.get(c, "") or "") for c in EVAL_ONLY_COLS}
            )

    # Delete checkpoint file after domain is complete
    if checkpoint_path.exists():
        checkpoint_path.unlink()
        print(f"  Checkpoint file removed: {checkpoint_path}")

# ============================================================
# STEP 5 — Write single combined output file
# ============================================================

combined_out = Path("CQ_EVALUATION_ONLY.csv")
with combined_out.open("w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=EVAL_ONLY_COLS, extrasaction="ignore")
    w.writeheader()
    for r in all_eval_only_rows:
        w.writerow(r)

print(f"\nFinal output saved: {combined_out.resolve()}")
print(f"Total rows: {len(all_eval_only_rows)}")

Running smoke test...
Smoke test passed.
CCO schema loaded: 16 classes, 14 object properties, 7 datatype properties.

Domain:      Finance
Pending:     252 | Evaluating now: 252
Input file:  Finance/cqs_finance_fixed.csv
  CQ001 (1/252, 0.4%): Yes - The clause can be represented using CCO's Regulation class with specifiesNo...
  CQ002 (2/252, 0.8%): Yes - The clause specifies a validity start date for guidelines, which can be rep...
  CQ003 (3/252, 1.2%): Yes - The clause explicitly states a supersession relationship between regulation...
  CQ004 (4/252, 1.6%): Yes - The clause specifies an obligation that applies to a role ("Competent autho...
  CQ005 (5/252, 2.0%): Yes - The clause contains an obligation with an action ("incorporating them into ...
  CQ006 (6/252, 2.4%): Yes - The clause contains an explicit obligation with a specific action, agent ro...
  CQ008 (7/252, 2.8%): Yes - The clause establishes an obligation for agents holding a specific role to ...
  CQ010 (8/252, 3.2%): 